# AWS Secrets Sync using Vault's local IRSA role

Doormat authenticates Terraform to Vault's AWS account. The Vault destination receives no static credentials and no target role, so the Vault pod uses its local IRSA credentials through the AWS SDK provider chain.

In [1]:
%env WORKDIR=/tmp/vault

env: WORKDIR=/tmp/vault


In [2]:
import os
from dotenv import load_dotenv

load_dotenv("./.env")

VAULT_TOKEN = os.getenv("VAULT_TOKEN")
VAULT_ADDR = os.getenv("VAULT_ADDR")
VAULT_CACERT = os.getenv("VAULT_CACERT")
AWS_REGION = os.getenv("AWS_REGION")

In [3]:
! vault status

Key                      Value
---                      -----
Seal Type                awskms
Recovery Seal Type       shamir
Initialized              true
Sealed                   false
Total Recovery Shares    1
Threshold                1
Version                  2.0.3+ent
Build Date               2026-06-16T21:32:56Z
Storage Type             raft
Cluster Name             vault-cluster-e89137a9
Cluster ID               b24418e0-d91d-d6d3-c571-5af49b1ec963
Removed From Cluster     false
HA Enabled               true
HA Cluster               https://vault-0.vault-internal:8201
HA Mode                  active
Active Since             2026-07-23T13:27:14.354277171Z
Raft Committed Index     13580
Raft Applied Index       13580
Last WAL                 5252


In [4]:
import os
import subprocess

# Authenticate Terraform to the AWS account that hosts Vault and its IRSA role.
subprocess.run(["doormat", "login", "-f"], check=True)
result = subprocess.run(
    ["bash", "-lc", 'eval "$(doormat aws -a aws_jose.merchan_test export)" && env -0'],
    check=True,
    capture_output=True,
)
for entry in result.stdout.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

# Prevent values from the static-account workflow overriding local IRSA mode.
os.environ.pop("TF_VAR_aws_access_key_id", None)
os.environ.pop("TF_VAR_aws_secret_access_key", None)

caller = subprocess.run(
    ["aws", "sts", "get-caller-identity", "--query", "{Account:Account,Arn:Arn}", "--output", "json"],
    check=True,
    capture_output=True,
    text=True,
)
print(caller.stdout)

time="2026-07-23T16:28:24+02:00" level=info msg="logging into doormat..."
time="2026-07-23T16:28:28+02:00" level=info msg="successfully logged into doormat!"


{
    "Account": "492487827579",
    "Arn": "arn:aws:sts::492487827579:assumed-role/aws_jose.merchan_test-developer/jose.merchan@hashicorp.com"
}



In [5]:
! vault write -f sys/activation-flags/secrets-sync/activate

Key            Value
---            -----
activated      [secrets-sync]
unactivated    [oauth-resource-server secrets-import]


In [6]:
! terraform -chdir=terraform-irsa init
! terraform -chdir=terraform-irsa validate

Initializing the backend...

Initializing provider plugins...
- Reusing previous version of hashicorp/aws from the dependency lock file
- Reusing previous version of hashicorp/vault from the dependency lock file
- Using previously-installed hashicorp/vault v5.10.1
- Using previously-installed hashicorp/aws v6.56.0


Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this command to reinitialize your working directory. If you forget, other
commands will detect it and remind you to do so if necessary.
Success! The configuration is valid.



In [7]:
! terraform -chdir=terraform-irsa apply -auto-approve

data.aws_iam_role.vault_irsa: Reading...
data.aws_region.current: Reading...
data.aws_caller_identity.current: Reading...
data.aws_region.current: Read complete after 0s [id=eu-central-1]
data.aws_caller_identity.current: Read complete after 0s [id=492487827579]
data.aws_iam_role.vault_irsa: Read complete after 0s [id=vault-kms-auto-unseal]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # aws_iam_role_policy.vault_secrets_sync will be created
  + resource "aws_iam_role_policy" "vault_secrets_sync" {
      + id          = (known after apply)
      + name        = "vault-secrets-sync-local"
      + name_prefix = (known after apply)
      + policy      = jsonencode(
            {
              + Statement = [
                  + {
                      + Action   = [
                          + "secretsmanager:CreateSecret",
            

In [8]:
! vault read sys/sync/destinations/aws-sm/aws-sm-irsa-local

Key                   Value
---                   -----
connection_details    map[region:eu-central-1]
name                  aws-sm-irsa-local
options               map[custom_tags:map[Managed_by:HashiCorp Vault] granularity_level:secret-path secret_name_template:vault_sync_{{ .SecretBaseName | lowercase }}]
type                  aws-sm


In [10]:
! vault kv get sync-aws-irsa/verification

Key                        Value
---                        -----
associated_secrets         map[kv_49ccfaec/test:map[accessor:kv_49ccfaec external_name:vault_sync_test last_operation:Write mount:kv secret_name:test sync_status:SYNCED updated_at:2026-07-23T14:29:43.086939332Z]]
store_name                 aws-sm-irsa-local
store_type                 aws-sm
sync_operation_counters    map[SYNCED:1]


In [11]:
! vault kv patch -mount=sync-aws-irsa verification sync_retry="$(date -u +%FT%TZ)"

== Secret Path ==
kv/data/test

======= Metadata =======
Key                Value
---                -----
created_time       2026-07-23T14:46:42.481326569Z
custom_metadata    <nil>
deletion_time      n/a
destroyed          false
version            2


In [16]:
! vault read -format=json \
  sys/sync/destinations/aws-sm/aws-sm-irsa-local/associations

{
  "request_id": "a9050573-622a-722b-9e66-a92b561955e4",
  "lease_id": "",
  "lease_duration": 0,
  "renewable": false,
  "data": {
    "associated_secrets": {},
    "store_name": "aws-sm-irsa-local",
    "store_type": "aws-sm",
    "sync_operation_counters": {
      "SYNCED": 1
    },
    "unsync_operation_counters": {
      "UNSYNCED": 1
    }
  },
  "warnings": null,
  "mount_type": "system"
}


# CLEAN UP

In [17]:
! terraform -chdir=terraform-irsa destroy -auto-approve

data.aws_iam_role.vault_irsa: Reading...
data.aws_region.current: Reading...
data.aws_caller_identity.current: Reading...
data.aws_region.current: Read complete after 0s [id=eu-central-1]
data.aws_caller_identity.current: Read complete after 0s [id=492487827579]
data.aws_iam_role.vault_irsa: Read complete after 0s [id=vault-kms-auto-unseal]
aws_iam_role_policy.vault_secrets_sync: Refreshing state... [id=vault-kms-auto-unseal:vault-secrets-sync-local]
vault_secrets_sync_aws_destination.local_irsa: Refreshing state... [id=aws-sm-irsa-local]

Terraform used the selected providers to generate the following execution plan.
Resource actions are indicated with the following symbols:
  - destroy

Terraform will perform the following actions:

  # aws_iam_role_policy.vault_secrets_sync will be destroyed
  - resource "aws_iam_role_policy" "vault_secrets_sync" {
      - id          = "vault-kms-auto-unseal:vault-secrets-sync-local" -> null
      - name        = "vault-secrets-sync-local" -> null
